# Natural Language Processing

Natural language processing (NLP) involves using computers to process and analyze human language. To work with text, we represent it numerically.

This notebook uses George Orwell’s *1984* to explore text extraction, tokenization, and numerical representations of text.

## Preparing the text

We begin by extracting text from a PDF, then examine ways to split it into tokens and represent those tokens numerically.


## Extracting text from a PDF

We use PyPDF2 to extract the text of George Orwell’s *1984* from a PDF. The extracted text becomes the input for the tokenization and representation examples that follow.

In [1]:
from pathlib import Path

import PyPDF2

repo_root = Path.cwd()
if repo_root.name == "examples":
    repo_root = repo_root.parent

pdf_path = repo_root / "data" / "orwell1984.pdf"

with pdf_path.open("rb") as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = "\n".join(
        page.extract_text() or ""
        for page in pdf_reader.pages
    )

print(f"Extracted {len(text):,} characters.")
print("\nText preview:\n")
print(text[:150])

Extracted 583,865 characters.

Text preview:

1984nineteen
eighty-four
a novelGeorge Orwell
NINETEEN EIGHTY-FOUR
ERICARTHUR BLAIR was born in British India in 1903, the son of a Civil Servant
work


## Tokens

Tokenization splits text into smaller units called tokens. Depending on the tokenizer, a token can be a word, part of a word, punctuation mark, number, or symbol.

In [2]:
tokens = text.split()

print(f"Number of whitespace-separated tokens: {len(tokens):,}")
print("\nFirst 10 tokens:")
print(tokens[:10])

Number of whitespace-separated tokens: 98,816

First 10 tokens:
['1984nineteen', 'eighty-four', 'a', 'novelGeorge', 'Orwell', 'NINETEEN', 'EIGHTY-FOUR', 'ERICARTHUR', 'BLAIR', 'was']


## Tokenizing with NLTK

Whitespace splitting leaves punctuation attached to words. We now use NLTK’s word tokenizer to separate words and punctuation into tokens.

In [3]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab", quiet=True)

nltk_tokens = word_tokenize(text)

print(f"Number of NLTK tokens: {len(nltk_tokens):,}")
print("\nFirst 10 tokens:")
print(nltk_tokens[:10])

/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/nick/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


Number of NLTK tokens: 117,709

First 10 tokens:
['1984nineteen', 'eighty-four', 'a', 'novelGeorge', 'Orwell', 'NINETEEN', 'EIGHTY-FOUR', 'ERICARTHUR', 'BLAIR', 'was']


## Recognizing names and places

Tokenization splits text into pieces. Named entity recognition adds labels to spans such as people, places, and organizations. Here, we use spaCy to find these in a short excerpt from *1984*.

In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(text[:3000])

print("Named entities:\n")

for entity in doc.ents:
    print(
        f"{entity.text:<30} "
        f"{entity.label_:<12} "
        f"{spacy.explain(entity.label_)}"
    )

Named entities:

1984nineteen                   CARDINAL     Numerals that do not fall under another type
British                        NORP         Nationalities or religious or political groups
India                          GPE          Countries, cities, states
1903                           DATE         Absolute or relative dates or periods
the Opium Department           ORG          Companies, agencies, institutions, etc.
Eton                           PERSON       People, including fictional
1922                           DATE         Absolute or relative dates or periods
the Indian Imperial Police     ORG          Companies, agencies, institutions, etc.
England                        GPE          Countries, cities, states
1927                           DATE         Absolute or relative dates or periods
the spring of 1928             DATE         Absolute or relative dates or periods
Paris                          GPE          Countries, cities, states
Blair                    

### About the output

Each row shows a span of text, its predicted entity label, and the label’s meaning. For example, `PERSON` identifies a person’s name, while `GPE` identifies a country, city, or state.

These are predictions based on context. Fictional names and places may be missed or assigned an incorrect label.

## Converting tokens into numbers

We can assign each unique token an integer ID. Repeated occurrences of the same token receive the same ID.

### Label encoding

Here, we use `LabelEncoder` to demonstrate this mapping. The IDs identify tokens; their numerical values do not represent meaning, similarity, or importance.

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
token_ids = le.fit_transform(nltk_tokens)

print(f"Number of tokens: {len(token_ids):,}")
print(f"Number of unique tokens: {len(le.classes_):,}")
print("\nFirst 10 tokens and their IDs:")

for token, token_id in zip(nltk_tokens[:10], token_ids[:10]):
    print(f"{token:<20} {token_id}")

Number of tokens: 117,709
Number of unique tokens: 15,325

First 10 tokens and their IDs:
1984nineteen         168
eighty-four          5410
a                    1864
novelGeorge          9244
Orwell               1278
NINETEEN             1167
EIGHTY-FOUR          698
ERICARTHUR           703
BLAIR                488
was                  14519


### About the output

Each token is paired with its integer ID. Repeated tokens receive the same ID, while different tokens receive different IDs.

These IDs are labels, not measurements: nearby numbers do not imply similar meanings. The same token also keeps its ID across different contexts, so this representation cannot distinguish its different uses.

### Same token, different meanings

“Orange” can refer to a fruit or a color:

- “She peeled an orange for breakfast.”
- “He painted the door orange.”

The code below highlights both occurrences and displays their token IDs. Label encoding assigns `orange` the same ID in both sentences, even though its meaning changes with context.

In [11]:
from IPython.display import HTML, display

sentences = [
    "She peeled an orange for breakfast.",
    "He painted the door orange.",
]

sentence_tokens = [word_tokenize(sentence) for sentence in sentences]

example_encoder = LabelEncoder()
example_encoder.fit([token for tokens in sentence_tokens for token in tokens])

for sentence, tokens in zip(sentences, sentence_tokens):
    token_ids = example_encoder.transform(tokens)
    orange_id = token_ids[tokens.index("orange")]

    highlighted = sentence.replace(
        "orange",
        '<strong style="color: #e67e22;">orange</strong>',
    )
    display(HTML(f"<p>{highlighted}<br>Token ID: <b>{orange_id}</b></p>"))

### About the output

Both highlighted occurrences of `orange` have the same token ID. In the first sentence, it refers to a fruit; in the second, a color.

The ID identifies the token but does not represent its meaning in context. Later, contextual embeddings will let us represent the same word differently depending on the surrounding text.

## TF-IDF

Token IDs identify words but do not measure their importance in a document. TF-IDF assigns weights based on how often a term appears within a document and how common it is across the collection.

Terms that occur frequently in one document but rarely in others tend to receive higher weights. This helps distinguish documents by their vocabulary, but it does not resolve meanings such as “orange” as a fruit versus a color.


In [12]:
import pandas as pd
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

documents = sent_tokenize(text)[:5]

tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(documents)

for i, sentence in enumerate(documents, 1):
    print(f"{i}: {sentence}\n")

pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[f"Sentence {i}" for i in range(1, len(documents) + 1)],
).round(3)

1: 1984nineteen
eighty-four
a novelGeorge Orwell
NINETEEN EIGHTY-FOUR
ERICARTHUR BLAIR was born in British India in 1903, the son of a Civil Servant
working in the Opium Department.

2: He attended Eton, but his family could not
pay for university, so in 1922 he joined the Indian Imperial Police.

3: He came tohate imperialism, and when he returned to England in 1927 he decided to resignand become a writer.

4: In the spring of 1928, he moved to Paris, hoping to make aliving as a freelance writer.

5: His lack of success reduced Blair to taking menialjobs as a dishwasher for a few weeks, principally in a fashionable hotel (theHotel X), which he later described in his ﬁ rst book, Down and Out in Paris andLondon.



,1903,1922,1927,1928,1984nineteen,aliving,andlondon,attended,blair,book,...,son,spring,success,taking,thehotel,tohate,university,weeks,working,writer
Sentence 1,0.226,0.000,0.00,0.000,0.226,0.000,0.00,0.000,0.182,0.00,...,0.226,0.000,0.00,0.00,0.00,0.00,0.000,0.00,0.226,0.000
Sentence 2,0.000,0.316,0.00,0.000,0.000,0.000,0.00,0.316,0.000,0.00,...,0.000,0.000,0.00,0.00,0.00,0.00,0.316,0.00,0.000,0.000
Sentence 3,0.000,0.000,0.34,0.000,0.000,0.000,0.00,0.000,0.000,0.00,...,0.000,0.000,0.00,0.00,0.00,0.34,0.000,0.00,0.000,0.274
Sentence 4,0.000,0.000,0.00,0.347,0.000,0.347,0.00,0.000,0.000,0.00,...,0.000,0.347,0.00,0.00,0.00,0.00,0.000,0.00,0.000,0.280
Sentence 5,0.000,0.000,0.00,0.000,0.000,0.000,0.24,0.000,0.194,0.24,...,0.000,0.000,0.24,0.24,0.24,0.00,0.000,0.24,0.000,0.000


### About the output

Each row represents a sentence from *1984*, and each column represents a term. The values are TF-IDF weights: higher values indicate terms that help distinguish that sentence from the others.

A zero means the term does not appear in that sentence. Common English stop words are excluded. These weights describe vocabulary patterns, not the meaning of a word in context.

## Dense token embeddings

An embedding represents a token as a vector of numbers. In trained models, these vectors can capture useful relationships between tokens.

We begin with randomly initialized vectors to show the structure of an embedding matrix. These vectors have not been trained, so their distances do not yet represent meaningful similarities.

A fixed embedding assigns each token the same vector wherever it appears. Contextual embeddings, such as those produced by BERT, can represent the same token differently depending on its surrounding text.

In [13]:
import numpy as np
import pandas as pd

sample_tokens = nltk_tokens[:10]
unique_tokens = list(dict.fromkeys(sample_tokens))

embedding_dim = 4
rng = np.random.default_rng(42)
embedding_matrix = rng.normal(
    size=(len(unique_tokens), embedding_dim)
)

pd.DataFrame(
    embedding_matrix,
    index=unique_tokens,
    columns=[f"Dimension {i}" for i in range(1, embedding_dim + 1)],
).round(3)

,Dimension 1,Dimension 2,Dimension 3,Dimension 4
1984nineteen,0.305,-1.040,0.750,0.941
eighty-four,-1.951,-1.302,0.128,-0.316
a,-0.017,-0.853,0.879,0.778
novelGeorge,0.066,1.127,0.468,-0.859
Orwell,0.369,-0.959,0.878,-0.050
NINETEEN,-0.185,-0.681,1.223,-0.155
EIGHTY-FOUR,-0.428,-0.352,0.532,0.365
ERICARTHUR,0.413,0.431,2.142,-0.406
BLAIR,-0.512,-0.814,0.616,1.129
was,-0.114,-0.840,-0.824,0.651


### About the output

Each row represents a unique token from the first 10 NLTK tokens, and each column is one dimension of its vector.

The values are randomly initialized, so nearby vectors do not indicate similar meanings. This demonstrates the structure of an embedding matrix; learning useful values requires training.